# Lab 1：静态页面与 API 数据采集

**目标**

1. 找到 `/devices` 页面的 API，分页采集全部 **5000** 条设备；
2. 用 BeautifulSoup 解析 `/companies` 静态 HTML，采集 **3 页共 60** 家企业；
3. 验证数量、ID 唯一性和 Company–Device 一致性；
4. 保存为 `data/devices.json` 与 `data/companies.json`。

> 运行方式：菜单 `Run → Run All Cells`。每做完一步对照手册检查输出。

## 任务一：Device —— 判断动态加载并找到 API

**判断依据**（手动在浏览器完成，用于写报告）：

- 在 `/devices` 右键 → **查看网页源代码**，`Ctrl+F` 搜索页面上的设备名称 → **搜不到**；
- `F12` → **Network** → **Fetch/XHR** → 刷新 → 找到返回设备列表的请求：

```text
Request URL : https://115.159.159.82/api/devices?page=1&page_size=50
Method      : GET
Query 参数   : page（页码）、page_size（每页条数，默认 50，上限 100）
```

→ 结论：**动态加载**（页面骨架是静态的，设备数据由 JS 调用 API 后再渲染）。

In [1]:
import requests

requests.packages.urllib3.disable_warnings()  # 屏蔽自签名证书告警

BASE_URL = "https://115.159.159.82"
api_url = BASE_URL + "/api/devices"

# 3.2 用 requests 复现第一页请求
r = requests.get(
    api_url,
    params={
        "page": 1,
        "page_size": 50,
    },
    timeout=10,
    verify=False,
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [3]:
# 读取 JSON，确认分页字段
data = r.json()

print(type(data))
print(data.keys())

print("当前页 page     :", data["page"])
print("每页条数 page_size:", data["page_size"])
print("设备总数 total   :", data["total"])
print("本页设备列表 items:", type(data["items"]), len(data["items"]), "条")

# 查看一条真实设备记录（了解字段含义）
print()
print(data["items"][0])

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])
当前页 page     : 1
每页条数 page_size: 50
设备总数 total   : 5000
本页设备列表 items: <class 'list'> 50 条

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [4]:
# 3.3 计算总页数（不要写死 100）
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
# 分页采集全部设备
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size,
        },
        timeout=10,
        verify=False,
    )

    page_data = r.json()

    # 将本页设备加入 devices
    devices.extend(page_data["items"])

print("device count:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

device count: 5000
unique device_id: 5000


## 任务二：Company —— 静态页面解析

**判断依据**：在 `/companies` 右键 → 查看网页源代码 → `Ctrl+F` 搜索企业名称 → **能搜到**，
说明企业数据随 HTML 一起返回，属于**静态加载**，可以直接用 `requests` + `BeautifulSoup` 解析。

**Selector 设计思路**：

```text
① 有 id 吗？没有
② 有语义明确的 class 吗？有 → article.company-card（每条企业记录的最外层节点）
③ data-* 属性？有 → data-company-id 直接给出 company_id
→ 结论：用 article.company-card，短、语义清晰、不依赖层级位置
```

In [15]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10, verify=False)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")

print("本页企业数：", len(cards))

本页企业数： 20


In [16]:
# 4.2 先解析第一条，确认字段提取方式
card = cards[0]

# dt/dd 结构先转成字典
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

company_id = card.get("data-company-id")            # 来自 HTML 属性
name = card.select_one("h2 a").get_text(strip=True)  # 来自文本
status = card.select_one(".badge").get_text(strip=True)

print(company_id, name, status)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}
C000001 临港精工001有限公司 运营中


In [17]:
# 把“一条记录”的解析逻辑写成函数，方便循环 3 页
import re


def parse_companies(soup):
    """从一页 HTML 中解析出企业记录列表"""
    result = []

    for card in soup.select("article.company-card"):
        details = {}
        for item in card.select("dl > div"):
            details[item.select_one("dt").get_text(strip=True)] = (
                item.select_one("dd").get_text(strip=True)
            )

        result.append(
            {
                "company_id": card.get("data-company-id"),
                "name": card.select_one("h2 a").get_text(strip=True),
                "industry": details.get("所属行业"),
                "zone": details.get("所在区域"),
                "device_count": int(re.sub(r"\D", "", details.get("设备数量", "0"))),
                "status": card.select_one(".badge").get_text(strip=True),
            }
        )

    return result


first_page = parse_companies(soup)
print("本页企业数：", len(first_page))
print(first_page[0])

本页企业数： 20
{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


## 任务三：根据“下一页”连续采集 3 页

页面底部的翻页区：

```html
<nav class="pagination">
  <span>上一页</span><b>1</b><a href="/companies?page=2">下一页</a>
</nav>
```

翻页区里 **最后一个 `<a>`** 永远是“下一页”（第 1 页时“上一页”是 `<span>`，第 2 页起才变成 `<a>`），
所以用 `nav.pagination a:last-child` 最稳定。

In [18]:
# 找“下一页”并读取 href
next_link = soup.select_one("nav.pagination a:last-child")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [19]:
from urllib.parse import urljoin

companies = []
current_url = base_url + "/companies"

for i in range(3):  # 连续采集 3 页
    # 1) 请求当前页
    r = requests.get(current_url, timeout=10, verify=False)
    soup = BeautifulSoup(r.text, "html.parser")

    # 2) 解析本页企业
    page_companies = parse_companies(soup)
    companies.extend(page_companies)
    print(f"第{i + 1}页：{len(page_companies)} 条")

    # 3) 找到下一页 href
    next_link = soup.select_one("nav.pagination a:last-child")
    if next_link is None:
        print("没有下一页了，停止")
        break
    current_url = urljoin(current_url, next_link.get("href"))

print("Company 总数：", len(companies))

第1页：20 条
第2页：20 条
第3页：20 条
Company 总数： 60


## 任务四：验证采集结果

目标：Company 60 条（60 个唯一 ID）、Device 5000 条（5000 个唯一 ID）、不一致企业数 = 0。

In [20]:
# 6.1 数量与 ID 唯一性
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))

print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


In [21]:
# 6.2 Company–Device 一致性 + 最终验证（截图 04_validation.png 就用这一格的输出）
from collections import Counter

device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)

    if expected != actual:
        mismatch_count += 1

# —— 最终验证结果（这 5 行必须同时出现在截图里）——
print("Company 总数        :", len(companies))
print("唯一 company_id 数   :", len(set(x["company_id"] for x in companies)))
print("Device 总数         :", len(devices))
print("唯一 device_id 数    :", len(set(x["device_id"] for x in devices)))
print("Company-Device 不一致企业数:", mismatch_count)

Company 总数        : 60
唯一 company_id 数   : 60
Device 总数         : 5000
唯一 device_id 数    : 5000
Company-Device 不一致企业数: 0


## 任务五：保存采集结果

统一保存到 Notebook 同级目录的 `data/` 下（本实验先只用 JSON）。

In [22]:
from pathlib import Path
import json

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")
print(sorted(p.name for p in data_dir.iterdir()))

数据已保存到 data/ 目录
['companies.json', 'devices.json']
